In [1]:
import importlib
import sys
import numpy as np
from collections import Counter
import pandas as pd

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')

In [2]:
# Load the data
# Path to your pickle file (saved with torch.save)
file_path_train = '../../../../../data/Helpdesk/tensor_data/normal/helpdesk_all_5_train.pkl'
# Load the dataset using torch.load
helpdesk_train_dataset = torch.load(file_path_train, weights_only=False)

# Path to your pickle file (saved with torch.save)
file_path_val = '../../../../../data/Helpdesk/tensor_data/normal/helpdesk_all_5_val.pkl'
# Load the dataset using torch.load
helpdesk_val_dataset = torch.load(file_path_val, weights_only=False)

In [3]:
# Helpdesk dataset dynamic categories, features:
helpdesk_all_categories = helpdesk_train_dataset.all_categories

helpdesk_all_categories_cat = helpdesk_all_categories[0]
helpdesk_all_categories_num = helpdesk_all_categories[1]
for i, cat in enumerate(helpdesk_all_categories_cat):
     print(f"Helpdesk dynamic categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(helpdesk_all_categories_num):
     print(f"Helpdesk dynamic numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk amount of numerical: {num[1]}")
print('\n')
     
# Helpdesk dataset static categories, features:
helpdesk_all_stat_categories = helpdesk_train_dataset.all_static_categories

helpdesk_all_stat_categories_cat = helpdesk_all_stat_categories[0]
helpdesk_all_stat_categories_num = helpdesk_all_stat_categories[1]
for i, cat in enumerate(helpdesk_all_stat_categories_cat):
     print(f"Helpdesk static categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(helpdesk_all_stat_categories_num):
     print(f"Helpdesk static numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk amount of numerical: {num[1]}")

Helpdesk dynamic categorical feature: Activity, Index position in categorical data list: 0
Helpdesk amount of category labels: 15
Helpdesk dynamic categorical feature: Resource, Index position in categorical data list: 1
Helpdesk amount of category labels: 24


Helpdesk dynamic numerical feature: case_elapsed_time, Index position in categorical data list: 0
Helpdesk amount of numerical: 1
Helpdesk dynamic numerical feature: event_elapsed_time, Index position in categorical data list: 1
Helpdesk amount of numerical: 1
Helpdesk dynamic numerical feature: day_in_week, Index position in categorical data list: 2
Helpdesk amount of numerical: 1
Helpdesk dynamic numerical feature: seconds_in_day, Index position in categorical data list: 3
Helpdesk amount of numerical: 1


Helpdesk static categorical feature: VariantIndex, Index position in categorical data list: 0
Helpdesk amount of category labels: 175
Helpdesk static categorical feature: seriousness, Index position in categorical data list:

In [4]:
# Create lists with name of Encoder features (input) and decoder features (input & output)

# Encoder features (fixed): use only requested features
enc_feat_cat = ['Activity']
enc_feat_num = ['case_elapsed_time']
enc_feat = [enc_feat_cat, enc_feat_num]
print("Input features encoder: ", enc_feat)

# Decoder features (activity only for T-GAN-LSTM)
dec_feat_cat = ['Activity']
dec_feat_num = []
dec_feat = [dec_feat_cat, dec_feat_num]
print("Features decoder: ", dec_feat)

Input features encoder:  [['Activity'], ['case_elapsed_time']]
Features decoder:  [['Activity'], []]


In [5]:
import suffix_pred.models.T_GAN_LSTM
importlib.reload(suffix_pred.models.T_GAN_LSTM)
from suffix_pred.models.T_GAN_LSTM import TaymouriAdversarialLSTM

# Prediction decoder output sequence length (fixed suffix length S)
seq_len_pred = helpdesk_train_dataset.min_suffix_size

# Size hidden layer
hidden_size = 50

# Number of LSTM cells
num_layers = 1

# STANDARD: One numerical output to predict
input_size = 1

# T-GAN-LSTM uses dynamic prefix features only
model_feat = enc_feat

# Output size: activity classes only
activity_feature_name = 'Activity'
activity_feature_index = next(i for i, cat in enumerate(helpdesk_all_categories_cat) if cat[0] == activity_feature_name)
output_size_act = helpdesk_all_categories_cat[activity_feature_index][1]

# concept_name_id: index of the activity feature in the categorical data list
concept_name_id = activity_feature_index

# T-GAN-LSTM model initialization
model = TaymouriAdversarialLSTM(
    data_set_categories=helpdesk_all_categories,
    model_feat=model_feat,
    concept_name_id=concept_name_id,
    hidden_size=hidden_size,
    num_layers=num_layers,
    seq_len_pred=seq_len_pred,
    input_size=input_size,
    output_size_act=output_size_act,
    dropout=0.2,
)

/home/PSPLab/.local/share/virtualenvs/decision_aware_augmentation_for_PPM-0DzgvVpC/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [6]:
import suffix_pred.loss
importlib.reload(suffix_pred.loss)
from suffix_pred.loss import Loss

loss_obj = Loss()

In [7]:
import suffix_pred.train
importlib.reload(suffix_pred.train)
from suffix_pred.train import TTraining

from torch.optim.lr_scheduler import ReduceLROnPlateau

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Start learning rate
learning_rate = 1e-5
weight_decay = 0.0

# Generator optimizer and scheduler
generator_optimizer = torch.optim.AdamW(params=model.seq2seq.parameters(), lr=learning_rate, weight_decay=weight_decay)
generator_scheduler = ReduceLROnPlateau(generator_optimizer, mode='min', factor=0.1, patience=15, min_lr=1e-8)

# Discriminator optimizer (for GAN mode)
discriminator_optimizer = torch.optim.AdamW(
    list(model.discriminator_lstm.parameters()) + list(model.discriminator_head.parameters()),
    lr=1e-3,
    weight_decay=weight_decay,
)

# Epochs / Batch size
num_epochs = 100
batch_size = 128

# Shuffle data
shuffle = True

# Teacher forcing schedule
max_teacher_forcing_value = 0.5
min_teacher_forcing_value = 0.0

# GAN training mode (set to False for MLE-only pre-training)
use_gan = True

optimize_values = {
    "optimizer": generator_optimizer,
    "generator_optimizer": generator_optimizer,
    "generator_scheduler": generator_scheduler,
    "discriminator_optimizer": discriminator_optimizer,
    "scheduler": generator_scheduler,
    "epochs": num_epochs,
    "mini_batches": batch_size,
    "shuffle": shuffle,
    "min_teacher_forcing_value": min_teacher_forcing_value,
    "max_teacher_forcing_value": max_teacher_forcing_value,
    "use_gan": use_gan,
    "lambda_adv": 0.1,
    "beam_width": 3,
}

# Suffix split value = fixed suffix length S
suffix_data_split_value = helpdesk_train_dataset.min_suffix_size
print("Suffix length S:", suffix_data_split_value)

# Activity feature EOS id
activity_label_to_id = helpdesk_all_categories_cat[concept_name_id][2]
eos_id = activity_label_to_id.get('EOS')
if eos_id is None:
    raise ValueError("Could not find EOS id in activity label mapping.")

model_output_path = "Helpdesk_T_GAN_LSTM_v1.pkl"

trainer = TTraining(
    device=device,
    model=model,
    data_train=helpdesk_train_dataset,
    data_val=helpdesk_val_dataset,
    optimize_values=optimize_values,
    suffix_data_split_value=suffix_data_split_value,
    concept_name_id=concept_name_id,
    eos_id=eos_id,
    loss_obj=loss_obj,
    save_model_n_th_epoch=1,
    saving_path=model_output_path,
)

# Train the model (GAN + MLE or MLE-only depending on use_gan flag)
train_gen_losses, train_disc_losses, val_losses, val_beam_token_acc = trainer.train()

Device: cuda
Suffix length S: 5


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100] - Mode: GAN, LR: 1e-05, TF: 0.5000, Gen Loss: 2.7041, Disc Loss: 0.9687, Val Loss: 2.6941, Beam Acc: 0.3281
saving model
Epoch [2/100] - Mode: GAN, LR: 1e-05, TF: 0.4800, Gen Loss: 2.6956, Disc Loss: 0.9354, Val Loss: 2.6867, Beam Acc: 0.3282
saving model
Epoch [3/100] - Mode: GAN, LR: 1e-05, TF: 0.4600, Gen Loss: 2.6879, Disc Loss: 0.9276, Val Loss: 2.6807, Beam Acc: 0.3282
saving model
Epoch [4/100] - Mode: GAN, LR: 1e-05, TF: 0.4400, Gen Loss: 2.6807, Disc Loss: 0.9262, Val Loss: 2.6728, Beam Acc: 0.3282
saving model
Epoch [5/100] - Mode: GAN, LR: 1e-05, TF: 0.4200, Gen Loss: 2.6728, Disc Loss: 0.9277, Val Loss: 2.6644, Beam Acc: 0.3282
saving model
Epoch [6/100] - Mode: GAN, LR: 1e-05, TF: 0.4000, Gen Loss: 2.6634, Disc Loss: 0.9289, Val Loss: 2.6550, Beam Acc: 0.3282
saving model
Epoch [7/100] - Mode: GAN, LR: 1e-05, TF: 0.3800, Gen Loss: 2.6530, Disc Loss: 0.9298, Val Loss: 2.6372, Beam Acc: 0.3282
saving model
Epoch [8/100] - Mode: GAN, LR: 1e-05, TF: 0.3600, Gen L